# Tuần 6 - Phần 2: Giả lập Thực chiến (Challenger vs Champion Simulation)

Sau khi chốt được mô hình AI Profit Targeting là tối ưu nhất trên giấy, bước cuối cùng trong vòng đời dự án Data Science là **Đem ra thực chiến** (A/B Test ngoài đời thực).

Trong Notebook này, chúng ta sẽ viết code giả lập một đợt tung Voucher A/B/C Test. Tập khách hàng sẽ được chia ngẫu nhiên làm 3 nhóm:
- **Holdout (10%):** Nhóm đối chứng, không bao giờ được nhận Voucher.
- **Champion (45%):** Nhóm chạy theo chiến lược cũ (Segment Targeting - Chỉ phát cho tệp Suburban).
- **Challenger (45%):** Nhóm chạy theo chiến lược AI mới (Profit Targeting - Ai sinh lãi mới phát).

Mục tiêu: Đánh giá xem nhóm Challenger có thực sự mang lại Lợi nhuận trung bình trên mỗi khách hàng (Profit per User) cao hơn nhóm Champion một cách có ý nghĩa thống kê (p-value < 0.05) hay không.

In [2]:
import pandas as pd
import numpy as np
import os
from scipy import stats

base_path = r"D:\Intern VSF\GSM-promotion-experimentation"
data_path = os.path.join(base_path, 'data', 'processed', 'test_predictions.csv')
df = pd.read_csv(data_path)

print("============================================================")
print("Feature 3: Challenger vs Champion Simulation")
print("============================================================")

Feature 3: Challenger vs Champion Simulation


## 1. Tái tạo lại hai trạng thái (Được nhận vs Không nhận)

In [3]:
# Để giả lập bất kỳ quyết định nào của Policy, ta cần biết Y0 (không voucher) và Y1 (có voucher) của từng người.
df['Y0'] = np.where(df['treatment_rand'] == 0, df['Y_rand'], df['Y_rand'] - df['cate_true'])
df['Y1'] = df['Y0'] + df['cate_true']

VOUCHER_RATE = 0.15
MARGIN_RATE  = 0.75
df['voucher_cost'] = df['avg_fare'] * VOUCHER_RATE
df['margin_per_ride'] = df['avg_fare'] * MARGIN_RATE

## 2. Phân bổ User vào 3 nhóm A/B/C Test

In [4]:
np.random.seed(99)
# Chia ngẫu nhiên: 10% Holdout, 45% Champion, 45% Challenger
rand_vals = np.random.rand(len(df))
df['experiment_group'] = np.where(rand_vals < 0.10, '1_Holdout',
                         np.where(rand_vals < 0.55, '2_Champion',
                                                    '3_Challenger'))

print("Số lượng khách hàng trong mỗi nhóm thực chiến:")
print(df['experiment_group'].value_counts())

Số lượng khách hàng trong mỗi nhóm thực chiến:
experiment_group
3_Challenger    1859
2_Champion      1767
1_Holdout        374
Name: count, dtype: int64


## 3. Thực thi Quyết định của từng Policy

In [5]:
def apply_policy(row):
    group = row['experiment_group']
    give_voucher = 0
    
    if group == '1_Holdout':
        give_voucher = 0  # Không bao giờ phát
        
    elif group == '2_Champion':
        # Luật cũ: Chỉ phát cho người Suburban
        if 'Suburban' in str(row['persona']):
            give_voucher = 1
            
    elif group == '3_Challenger':
        # Luật mới: Phát nếu AI dự báo Lãi ròng > 0
        expected_value = (row['cate_pred'] * row['margin_per_ride']) - (row['pred_rides_treated'] * row['voucher_cost'])
        if expected_value > 0:
            give_voucher = 1
            
    return give_voucher

df['actual_treatment_given'] = df.apply(apply_policy, axis=1)

# Tính toán kết quả số chuyến thực tế (Y_actual) và Lợi nhuận gộp sinh ra cho Công ty
df['Y_actual'] = np.where(df['actual_treatment_given'] == 1, df['Y1'], df['Y0'])
df['profit_generated'] = (df['Y_actual'] * df['margin_per_ride']) - (df['actual_treatment_given'] * df['Y_actual'] * df['voucher_cost'])

## 4. Báo cáo Kết quả A/B/C Test

In [6]:
summary = df.groupby('experiment_group').agg(
    Users=('Y_actual', 'count'),
    Vouchers_Sent=('actual_treatment_given', 'sum'),
    Total_Profit=('profit_generated', 'sum'),
    Avg_Profit_per_User=('profit_generated', 'mean')
).reset_index()

display(summary)

# Tính toán Statistical Significance (T-Test) giữa Challenger và Champion
champion_profits = df[df['experiment_group'] == '3_Challenger']['profit_generated']
challenger_profits = df[df['experiment_group'] == '2_Champion']['profit_generated']

t_stat, p_val = stats.ttest_ind(champion_profits, challenger_profits)
print(f"\nKiểm định T-Test giữa Challenger (AI) và Champion (K-Means):")
print(f"P-value = {p_val:.5f}")

if p_val < 0.05:
    print("✅ KẾT LUẬN: Lợi nhuận của Challenger vượt trội hơn Champion một cách CÓ Ý NGHĨA THỐNG KÊ (P-value < 0.05)!")
    print("Đây là bằng chứng vững chắc nhất để thuyết phục Ban Giám đốc Roll-out mô hình AI ra toàn hệ thống.")
else:
    print("❌ Lợi nhuận chưa có sự khác biệt rõ rệt về mặt thống kê. Cần chạy test thêm thời gian.")

,experiment_group,Users,Vouchers_Sent,Total_Profit,Avg_Profit_per_User
0,1_Holdout,374,0,54772.459629,146.450427
1,2_Champion,1767,765,256090.514870,144.929550
2,3_Challenger,1859,430,272317.454018,146.485989



Kiểm định T-Test giữa Challenger (AI) và Champion (K-Means):
P-value = 0.80522
❌ Lợi nhuận chưa có sự khác biệt rõ rệt về mặt thống kê. Cần chạy test thêm thời gian.
